# YOLOv8 Manufacturing Defect Detection — NEU Surface Defect Dataset

Trains YOLOv8 for real-time surface defect detection on steel manufacturing surfaces.

**Dataset:** NEU Surface Defect Database — 1,800 images, 6 defect classes  
**GPU:** T4 (free Colab) — 100 epochs ~45 minutes  
**Author:** Md Arifur Rahman | github.com/arifme071

**Related research:** HMM-RL for WAAM Manufacturing (Springer 2026, Georgia Tech)

In [ ]:
# Install dependencies
!pip install ultralytics kaggle -q

In [ ]:
import torch
from ultralytics import YOLO
import os

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Download NEU Surface Defect Dataset

Option A: Kaggle (recommended)
Option B: Manual download from http://faculty.neu.edu.cn/yunhyan/NEU_surface_defect_database.html

In [ ]:
# Option A — Kaggle download
# First: upload your kaggle.json API key to Colab
from google.colab import files
# files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || echo 'No kaggle.json found — use manual download'
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database 2>/dev/null || echo 'Kaggle download failed'
!unzip -q neu-surface-defect-database.zip -d NEU-DET 2>/dev/null || echo 'Unzip failed'

In [ ]:
# Prepare dataset in YOLO format
import os, shutil, random
from pathlib import Path

CLASSES = ['Crazing', 'Inclusion', 'Patches', 'Pitted_surface', 'Rolled-in_scale', 'Scratches']
CLASS_NAMES = ['Crazing', 'Inclusion', 'Patches', 'Pitted Surface', 'Rolled-in Scale', 'Scratches']

random.seed(42)

# Create YOLO directory structure
for split in ['train', 'val', 'test']:
    os.makedirs(f'neu_yolo/images/{split}', exist_ok=True)
    os.makedirs(f'neu_yolo/labels/{split}', exist_ok=True)

# Collect all images
all_files = []
img_dir = Path('NEU-DET/NEU-DET/IMAGES') if Path('NEU-DET/NEU-DET/IMAGES').exists() else Path('NEU-DET/IMAGES')

for class_id, class_name in enumerate(CLASSES):
    images = list(img_dir.glob(f'{class_name}*.jpg'))
    if not images:
        images = list(img_dir.glob(f'{class_name.replace("-","_")}*.jpg'))
    for img in images:
        all_files.append((img, class_id))

print(f'Total images found: {len(all_files)}')

# Split 80/10/10
random.shuffle(all_files)
n = len(all_files)
n_train, n_val = int(n*0.8), int(n*0.1)
splits = {'train': all_files[:n_train], 'val': all_files[n_train:n_train+n_val], 'test': all_files[n_train+n_val:]}

for split_name, files in splits.items():
    for img_path, class_id in files:
        shutil.copy2(img_path, f'neu_yolo/images/{split_name}/{img_path.name}')
        # Full-image bounding box (classification task)
        label = f'{class_id} 0.5 0.5 1.0 1.0\n'
        with open(f'neu_yolo/labels/{split_name}/{img_path.stem}.txt', 'w') as f:
            f.write(label)
    print(f'  {split_name}: {len(files)} images')

In [ ]:
# Write dataset YAML
yaml_content = f"""path: /content/neu_yolo
train: images/train
val: images/val
test: images/test

nc: 6
names: ['Crazing', 'Inclusion', 'Patches', 'Pitted Surface', 'Rolled-in Scale', 'Scratches']
"""

with open('neu_dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('Dataset YAML written: neu_dataset.yaml')

## 2. Train YOLOv8

In [ ]:
model = YOLO('yolov8s.pt')  # start from COCO pretrained weights

results = model.train(
    data='neu_dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='runs',
    name='neu_defect',
    # Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5, mosaic=1.0,
    # Optimizer
    optimizer='AdamW', lr0=0.001,
    patience=20,
    verbose=True,
)

print(f'Best mAP@50: {results.results_dict["metrics/mAP50(B)"]:.4f}')

## 3. Evaluate

In [ ]:
best_model = YOLO('runs/neu_defect/weights/best.pt')
metrics = best_model.val(data='neu_dataset.yaml', split='test')

print('\n── Test Set Results ──')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
print('\nPer-class AP@50:')
classes = ['Crazing', 'Inclusion', 'Patches', 'Pitted Surface', 'Rolled-in Scale', 'Scratches']
for cls, ap in zip(classes, metrics.box.ap50):
    print(f'  {cls:<20}: {ap:.4f}')

## 4. Export to Intel OpenVINO

In [ ]:
# Export to OpenVINO IR format for Intel hardware deployment
best_model.export(format='openvino', imgsz=640)
print('Model exported to OpenVINO IR format')
print('Optimized for Intel CPUs, GPUs, and VPUs')
!ls runs/neu_defect/weights/

## 5. Run inference on sample images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Run on test images
test_images = list(Path('neu_yolo/images/test').glob('*.jpg'))[:6]
results = best_model(test_images, conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (result, ax) in enumerate(zip(results, axes.flat)):
    annotated = result.plot()
    ax.imshow(annotated[:, :, ::-1])
    ax.set_title(f'Image {i+1}: {Path(result.path).stem}', fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8 Surface Defect Detection — NEU Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('detection_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: detection_results.png')